# Random Survival Forest Modeling Workflow

## Companion workflow for

Detka et al. (2026)

*A Plant-Level Survival Modeling Framework for Spatiotemporal Strawberry Canopy Decline Using UAV Multispectral Time Series*

---

This notebook converts plant-level canopy metrics generated during the ArcGIS preprocessing workflow into longitudinal datasets, trains leakage-safe Random Survival Forest models, and exports plant-level risk predictions.

This notebook represents the second stage of the complete workflow.

01 ArcGIS preprocessing
        ↓
02 Random Survival Forest modeling
        ↓
03 Post-processing and figures

Step A — Build per-plant change features (safe)

This part is fine to do on the full dataset because it uses only each plant’s own history.

In [ ]:
from config import *

import pandas as pd
import numpy as np

df = pd.read_csv(PROCESSED_DATA / "ERL_2023_Field2_Merged_HexMetrics.csv")

df["flight_date"] = pd.to_datetime(df["flight_date"], errors="coerce")
df = df.sort_values(["Plant_Id", "flight_date"])

df["PDCnpA_change"] = df.groupby("Plant_Id")["PDCnpA"].diff().fillna(0)
df["PctDieback_change"] = df.groupby("Plant_Id")["PctDieback"].diff().fillna(0)
df["PctHlthy_change"] = df.groupby("Plant_Id")["PctHlthy"].diff().fillna(0)

Step B — Split by Plant_Id (do this before clustering)

In [13]:
from sklearn.model_selection import train_test_split

plant_ids = df["Plant_Id"].unique()
train_ids, test_ids = train_test_split(plant_ids, test_size=0.2, random_state=42)

df["split"] = np.where(df["Plant_Id"].isin(train_ids), "train", "test")

Step C — Per-flight clustering fit on TRAIN only, then label everyone. 
This produces the event label (Critical) in a way that the test set cannot influence how the labeler is defined.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

feature_cols = [
    "PDCnpA", "PctDieback", "PctHlthy",
    "PDCnpA_change", "PctDieback_change", "PctHlthy_change"
]

out = []

for flight_date, g in df.groupby("flight_date"):
    g = g.copy()
    g_train = g[g["split"] == "train"]

    # Fit labeler on TRAIN ONLY
    scaler = StandardScaler()
    X_train = scaler.fit_transform(g_train[feature_cols])

    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    kmeans.fit(X_train)

    # Assign clusters to ALL plants using TRAIN-fitted scaler + kmeans
    X_all = scaler.transform(g[feature_cols])
    g["Cluster"] = kmeans.predict(X_all)

    # Identify "critical" cluster using TRAIN ONLY
    g_train = g_train.copy()
    g_train["Cluster"] = kmeans.predict(X_train)

    cluster_means = g_train.groupby("Cluster")["PDCnpA"].mean()
    worst_cluster = cluster_means.idxmax()

    g["Critical_Cluster"] = worst_cluster
    g["Critical"] = (g["Cluster"] == worst_cluster).astype(int)

    out.append(g)

df_lsafe = pd.concat(out, ignore_index=True)
df_lsafe.to_csv(
    FIELD_OUTPUT_DIR / "combined_data_clustered_with_critical_per_flight_LEAKSAFE.csv",
    index=False
)
# optional summary
critical_summary = (
    df_lsafe.groupby("flight_date")["Critical"]
    .sum()
    .reset_index()
    .rename(columns={"Critical": "Number of Critical Plants"})
)
critical_summary.to_csv("critical_summary_LEAKSAFE.csv", index=False)

Step D — RSF formatting 

In [ ]:
df = pd.read_csv(
    FIELD_OUTPUT_DIR / "combined_data_clustered_with_critical_per_flight_LEAKSAFE.csv"
)

df["flight_date"] = pd.to_datetime(df["flight_date"], errors="coerce")

df["event"] = df["Critical"].astype(int)

df["time"] = df.groupby("Plant_Id")["flight_date"].transform(
    lambda x: (x - x.min()).dt.days
)

df.to_csv(
    FIELD_OUTPUT_DIR / "rsf_formatted_data_LEAKSAFE.csv",
    index=False
)

Step E -- Leakage-safe Random Survival Forest (RSF)
Expanding-window evaluation with per-flight stratified downsampling.

In [ ]:
"""
Leakage-safe Random Survival Forest (RSF)
Expanding-window evaluation with per-flight stratified downsampling.

Key changes vs. older script:
✅ Loads rsf_formatted_data_LEAKSAFE.csv
✅ Uses precomputed split (DO NOT regenerate it here)
✅ Uses event + time already in the leak-safe dataset
✅ Computes per-flight Test C-index (survival concordance)
✅ Saves per-flight test predictions for mapping
"""

import pandas as pd
import numpy as np
import joblib
import os

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored  

from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve
from sklearn.inspection import permutation_importance


# ---------------------------
# Load dataset (LEAK-SAFE)
# ---------------------------
df = pd.read_csv(FIELD_OUTPUT_DIR / "rsf_formatted_data_LEAKSAFE.csv")

# Parse flight_date
df["flight_date"] = pd.to_datetime(df["flight_date"], errors="coerce")

# Optional: restrict to your modeling window (matches your prior script)
df = df[df["flight_date"] >= "2023-02-09"].copy()

# Sort for readability / stability
df = df.sort_values(by=["flight_date", "Plant_Id"]).reset_index(drop=True)

# Sanity checks
required_cols = ["split", "event", "time", "flight_date", "Plant_Id", "Easting", "Northing"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in LEAKSAFE file: {missing}")

# ---------------------------
# Define features
# ---------------------------
features = ["PDCnpA", "PDCnpA_change", "PctDieback_change", "PctHlthy_change"]
missing_feat = [c for c in features if c not in df.columns]
if missing_feat:
    raise ValueError(f"Missing feature columns: {missing_feat}")

# ---------------------------
# Setup outputs
# ---------------------------
MODEL_DIR.mkdir(parents=True, exist_ok=True)
performance_results = []
predictions_list = []

flight_dates = sorted(df["flight_date"].dropna().unique())

# ---------------------------
# Expanding-window evaluation
# ---------------------------
for i in range(2, len(flight_dates)):
    train_dates = flight_dates[:i]
    test_date = flight_dates[i]

    print(f"\n🚀 Training on {len(train_dates)} flights, Testing on: {pd.to_datetime(test_date).date()}")

    train_df = df[(df["flight_date"].isin(train_dates)) & (df["split"] == "train")].copy()
    test_df  = df[(df["flight_date"] == test_date) & (df["split"] == "test")].copy()

    if test_df.empty:
        print(f"⚠️ Skipping {test_date}, no test data available.")
        continue
    # 🔎 investigate C inflation
    print("   time unique:", test_df["time"].nunique())
    print("   event counts:\n", test_df["event"].value_counts(dropna=False))
    # ---------------------------
    # Stratified downsampling per flight (train only)
    # ---------------------------
    sampled_train = []
    for flight in train_df["flight_date"].unique():
        g = train_df[train_df["flight_date"] == flight]
        crit = g[g["event"] == 1]
        healthy = g[g["event"] == 0].sample(frac=0.4, random_state=42)

        sampled_train.append(pd.concat([crit, healthy], axis=0))

    train_df = pd.concat(sampled_train, axis=0).sort_values(by=["flight_date", "Plant_Id"]).reset_index(drop=True)

    # ---------------------------
    # Prepare survival objects
    # ---------------------------
    y_train = Surv.from_arrays(train_df["event"].astype(bool), train_df["time"].astype(float))
    X_train = train_df[features]

    y_test  = Surv.from_arrays(test_df["event"].astype(bool), test_df["time"].astype(float))
    X_test  = test_df[features]

    # ---------------------------
    # Fit RSF
    # ---------------------------
    rsf = RandomSurvivalForest(
        n_estimators=50,
        max_depth=2,
        min_samples_split=40,
        min_samples_leaf=20,
        max_features=2,
        n_jobs=-1,
        random_state=42
    )
    rsf.fit(X_train, y_train)

    # ---------------------------
    # Permutation importance (optional)
    # ---------------------------
    perm = permutation_importance(rsf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
    print("\n🌿 Permutation Feature Importances:")
    for name, imp in zip(features, perm.importances_mean):
        print(f"{name}: {imp:.4f}")

    # Save model for this test_date
    model_path = MODEL_DIR / f"rsf_model_{pd.to_datetime(test_date).strftime('%Y-%m-%d')}.pkl"

    # ---------------------------
    # Predict risk scores (mean cumulative hazard)
    # ---------------------------
    chf_train = rsf.predict_cumulative_hazard_function(X_train)
    risk_scores_train = np.array([np.mean(fn.y) if len(fn.y) else 0 for fn in chf_train])

    chf_test = rsf.predict_cumulative_hazard_function(X_test)
    risk_scores_test = np.array([np.mean(fn.y) if len(fn.y) else 0 for fn in chf_test])

    # ---------------------------
    # ✅ C-index (survival ranking metric)
    # NOTE: concordance_index_censored expects higher score = higher risk.
    # Mean cumulative hazard is already "higher = worse", so this is aligned.
    # ---------------------------
    c_index = concordance_index_censored(
        test_df["event"].astype(bool).values,
        test_df["time"].astype(float).values,
        risk_scores_test
    )[0]

    # ---------------------------
    # Thresholding using F1 on THIS test flight
    # ---------------------------
    y_train_binary = train_df["event"].astype(int).values
    y_test_binary  = test_df["event"].astype(int).values

    precisions, recalls, thresholds = precision_recall_curve(y_test_binary, risk_scores_test)

    # thresholds has length = len(precisions)-1, so align carefully
    f1_scores = (2 * precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-12)
    best_idx = int(np.argmax(f1_scores)) if len(f1_scores) else 0
    best_threshold = thresholds[best_idx] if len(thresholds) else 0.0

    y_test_pred = (risk_scores_test > best_threshold).astype(int)
    y_train_pred = (risk_scores_train > best_threshold).astype(int)

    # Metrics
    precision_train = precision_score(y_train_binary, y_train_pred, zero_division=0)
    recall_train = recall_score(y_train_binary, y_train_pred, zero_division=0)
    f1_train = f1_score(y_train_binary, y_train_pred, zero_division=0)

    precision_test = precision_score(y_test_binary, y_test_pred, zero_division=0)
    recall_test = recall_score(y_test_binary, y_test_pred, zero_division=0)
    f1_test = f1_score(y_test_binary, y_test_pred, zero_division=0)

    performance_results.append({
        "Flight Date": pd.to_datetime(test_date),
        "Best Threshold": float(best_threshold),
        "Train Precision": precision_train,
        "Train Recall": recall_train,
        "Train F1": f1_train,
        "Test Precision": precision_test,
        "Test Recall": recall_test,
        "Test F1": f1_test,
        "Test C-index": float(c_index)  # ✅ ADD THIS
    })

    print(
        f"✅ {pd.to_datetime(test_date).date()} | "
        f"Train F1={f1_train:.2f} | Test F1={f1_test:.2f} | "
        f"P={precision_test:.2f}, R={recall_test:.2f} | C={c_index:.2f}"
    )

    # Save test predictions for mapping
    out = test_df[["Plant_Id", "flight_date", "Northing", "Easting", "time", "event"]].copy()
    out["risk_score"] = risk_scores_test
    out["risk_cindex_context"] = c_index  # optional: handy for debugging/record-keeping
    out["Test Flight Date"] = pd.to_datetime(test_date)
    predictions_list.append(out)

# ---------------------------
# Save outputs
# ---------------------------
df_perf = pd.DataFrame(performance_results)
df_perf.to_csv(FIELD_OUTPUT_DIR / "rsf_performance.csv", index=False)
print("\n🔥 RSF Performance Results Saved to rsf_performance.csv")

pred_df = pd.concat(predictions_list, ignore_index=True) if predictions_list else pd.DataFrame()
pred_df.to_csv(FIELD_OUTPUT_DIR / "rsf_test_predictions.csv", index=False)
print("📍 Test predictions saved to rsf_test_predictions.csv")

# ---------------------------
# Overall C-index across ALL test predictions
# ---------------------------
from sksurv.metrics import concordance_index_censored

if not pred_df.empty:
    c_overall = concordance_index_censored(
        pred_df["event"].astype(bool).values,
        pred_df["time"].astype(float).values,
        pred_df["risk_score"].astype(float).values
    )[0]

    print(f"\n📊 Overall Test C-index (all flights combined): {c_overall:.4f}")

    overall_df = pd.DataFrame({
        "Metric": ["Overall Test C-index"],
        "Value": [c_overall]
    })

    overall_df.to_csv(FIELD_OUTPUT_DIR / "rsf_overall_metrics.csv", index=False)
    print("✅ Overall C-index saved to rsf_overall_metrics.csv")

else:
    print("\n⚠️ pred_df is empty; cannot compute overall C-index.")


🚀 Training on 2 flights, Testing on: 2023-04-21
   time unique: 1
   event counts:
 event
0    10546
1      353
Name: count, dtype: int64

🌿 Permutation Feature Importances:
PDCnpA: 0.0740
PDCnpA_change: 0.0219
PctDieback_change: 0.0014
PctHlthy_change: 0.0019
✅ 2023-04-21 | Train F1=0.94 | Test F1=0.94 | P=0.91, R=0.96 | C=1.00

🚀 Training on 3 flights, Testing on: 2023-05-15
   time unique: 1
   event counts:
 event
0    10490
1      409
Name: count, dtype: int64

🌿 Permutation Feature Importances:
PDCnpA: 0.0937
PDCnpA_change: 0.0277
PctDieback_change: 0.0251
PctHlthy_change: -0.0023
✅ 2023-05-15 | Train F1=0.91 | Test F1=0.85 | P=0.90, R=0.80 | C=0.97

🚀 Training on 4 flights, Testing on: 2023-05-22
   time unique: 1
   event counts:
 event
0    10842
1       57
Name: count, dtype: int64

🌿 Permutation Feature Importances:
PDCnpA: 0.3701
PDCnpA_change: 0.0027
PctDieback_change: 0.0141
PctHlthy_change: -0.0085
✅ 2023-05-22 | Train F1=0.85 | Test F1=0.31 | P=0.28, R=0.35 | C=0.93

🚀

In [4]:
# ---------------------------
# Compute overall survival C-index (true survival metric)
# ---------------------------
from sksurv.metrics import concordance_index_censored

if not pred_df.empty:
    c_index_overall = concordance_index_censored(
        pred_df["event"].astype(bool).values,
        pred_df["time"].values,
        pred_df["risk_score"].values
    )[0]

    print("\n📊 Overall Test C-index (all flights combined):", round(c_index_overall, 4))
else:
    print("\n⚠️ No test predictions available for C-index computation.")


📊 Overall Test C-index (all flights combined): 0.8755


Step F - Apply trained RSF models to all plants per flight and export per-flight shapefiles.

In [ ]:
"""
Apply trained RSF models to all plants per flight and export per-flight shapefiles.

- Loads trained models from rsf_models/
- Applies each model to all plants on its corresponding flight date
- Saves rsf_predictions_all_plants.csv (full predictions)
- Saves per-flight shapefiles with normalized risk (within-flight)
"""

import os
import pandas as pd
import numpy as np
import geopandas as gpd
import joblib

# --- Config ---
input_csv = FIELD_OUTPUT_DIR / "rsf_formatted_data_LEAKSAFE.csv"
model_dir = MODEL_DIR
shapefile_dir = RISK_EXPORT_DIR
shapefile_dir.mkdir(parents=True, exist_ok=True)

# Your stated CRS (UTM 11N). Keep this consistent with Easting/Northing.
crs_code = "EPSG:32611"

# Features must match what the RSF models were trained on
features = ["PDCnpA", "PDCnpA_change", "PctDieback_change", "PctHlthy_change"]

# --- Load full data ---
df = pd.read_csv(input_csv)

# Ensure datetime + normalize to midnight for reliable matching
df["flight_date"] = pd.to_datetime(df["flight_date"], errors="coerce").dt.normalize()
df = df.dropna(subset=["flight_date"])

# Optional: filter to match your modeling window
df = df[df["flight_date"] >= pd.Timestamp("2023-02-09")].copy()
df = df.sort_values(by=["flight_date", "Plant_Id"])

# Confirm required columns exist
required_cols = set(features + ["Plant_Id", "Easting", "Northing", "event", "time", "flight_date"])
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in input CSV: {sorted(missing)}")

inference_results = []

# --- Loop through models ---
model_files = sorted([f for f in os.listdir(model_dir) if f.endswith(".pkl")])
if not model_files:
    raise FileNotFoundError(f"No .pkl models found in: {model_dir}")

for model_file in model_files:
    date_str = model_file.replace("rsf_model_", "").replace(".pkl", "")
    model_date = pd.to_datetime(date_str, errors="coerce")

    if pd.isna(model_date):
        print(f"⚠️ Could not parse date from model filename: {model_file} — skipping.")
        continue

    flight_date = model_date.normalize()
    print(f"\n🔮 Predicting all plants on: {flight_date.date()} ({model_file})")

    model_path = model_dir / model_file
    rsf = joblib.load(model_path)

    flight_df = df[df["flight_date"] == flight_date].copy()
    if flight_df.empty:
        print("⚠️ No data for this flight date — skipping.")
        continue

    print(f"   rows in flight_df: {len(flight_df):,}")
    print(f"   events in flight_df:\n{flight_df['event'].value_counts(dropna=False)}")

    X_flight = flight_df[features]

    # Predict risk scores
    try:
        chf_all = rsf.predict_cumulative_hazard_function(X_flight)
        # Use mean CHF as a scalar risk score (same as your training script)
        risk_scores = np.array([float(np.mean(fn.y)) if len(fn.y) else 0.0 for fn in chf_all])
    except Exception as e:
        print(f"❌ Error during prediction on {flight_date.date()}: {e}")
        risk_scores = np.zeros(len(X_flight), dtype=float)

    flight_df["risk_raw"] = risk_scores

    # Normalize risk within this flight (0–1)
    rmin = float(np.min(risk_scores))
    rmax = float(np.max(risk_scores))
    denom = (rmax - rmin) if (rmax - rmin) != 0 else 1.0
    flight_df["risk_norm"] = (risk_scores - rmin) / denom

    # Store rows for combined CSV
    # (keep your original names here; shapefile export will rename)
    inference_results.append(flight_df[[
        "Plant_Id", "flight_date", "Northing", "Easting",
        "time", "event", "risk_raw", "risk_norm"
    ]])

    # --- Convert to GeoDataFrame ---
    gdf = gpd.GeoDataFrame(
        flight_df,
        geometry=gpd.points_from_xy(flight_df["Easting"], flight_df["Northing"]),
        crs=crs_code
    )

    # --- Shapefile-safe field names (<=10 chars) ---
    # Shapefile truncates silently; better to control it explicitly.
    export_gdf = gdf[[
        "Plant_Id", "risk_raw", "risk_norm", "event", "geometry"
    ]].rename(columns={
        "Plant_Id": "plant_id",   # 8 chars
        "risk_raw": "risk_raw",   # 8 chars
        "risk_norm": "risk_norm", # 9 chars
        "event": "event"          # 5 chars
    })

    # --- Save shapefile ---
    out_name = f"all_risk_{flight_date.strftime('%y%m%d')}.shp"
    out_path = shapefile_dir / out_name
    export_gdf.to_file(out_path)

print("\n✅ All shapefiles exported to:", shapefile_dir)

# --- Combine and export CSV ---
if inference_results:
    all_df = pd.concat(inference_results, ignore_index=True)
    all_df.to_csv(FIELD_OUTPUT_DIR / "rsf_predictions_all_plants.csv", index=False)
    print("✅ Full risk predictions saved to rsf_predictions_all_plants.csv")
else:
    print("⚠️ No inference results were generated; combined CSV not written.")


🔮 Predicting all plants on: 2023-04-21 (rsf_model_2023-04-21.pkl)
   rows in flight_df: 52,420
   events in flight_df:
event
0    51349
1     1071
Name: count, dtype: int64

🔮 Predicting all plants on: 2023-05-15 (rsf_model_2023-05-15.pkl)
   rows in flight_df: 52,420
   events in flight_df:
event
0    51746
1      674
Name: count, dtype: int64

🔮 Predicting all plants on: 2023-05-22 (rsf_model_2023-05-22.pkl)
   rows in flight_df: 52,420
   events in flight_df:
event
0    49689
1     2731
Name: count, dtype: int64

🔮 Predicting all plants on: 2023-06-01 (rsf_model_2023-06-01.pkl)
   rows in flight_df: 52,420
   events in flight_df:
event
0    51428
1      992
Name: count, dtype: int64

🔮 Predicting all plants on: 2023-06-10 (rsf_model_2023-06-10.pkl)
   rows in flight_df: 52,420
   events in flight_df:
event
0    50232
1     2188
Name: count, dtype: int64

🔮 Predicting all plants on: 2023-06-20 (rsf_model_2023-06-20.pkl)
   rows in flight_df: 52,420
   events in flight_df:
event
0   